<a href="https://colab.research.google.com/github/megankomacho-boop/-/blob/main/API_%E4%BA%BA%E5%B7%A5%E8%A1%9B%E6%98%9F%E3%81%AE%E3%82%B7%E3%83%9F%E3%83%A5%E3%83%AC%E3%83%BC%E3%82%B7%E3%83%A7%E3%83%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install skyfield plotly

from skyfield.api import load
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"


SATS = {
    "だいち2号 (ALOS-2)": 39766,
    "だいち4号 (ALOS-4)": 60182,
    "いぶき2号 (GOSAT-2)": 43672,
    "しきさい (GCOM-C)": 43065,
    "XRISM": 57800,
    "ひまわり9号": 41836,
    "みちびき1号 (QZS-1)": 37158,
    "みちびき2号 (QZS-2)": 42738,
    "みちびき3号 (QZS-3)": 42917,
    "ISS": 25544,
}
#ひまわり・みちびき3号（静止衛星）は赤道上の1点からほぼ動きません。

ts = load.timescale()
minutes = 600
step = 1  # 1分刻み

now = ts.now().utc_datetime()
t = ts.utc(now.year, now.month, now.day, now.hour,
           range(now.minute, now.minute + minutes, step))

# 全衛星の軌跡を先に計算
data = {}
for name, catnr in SATS.items():
    try:
        url = f"https://celestrak.org/NORAD/elements/gp.php?CATNR={catnr}&FORMAT=TLE"
        #sat = load.tle_file(url)[0]
        sat = load.tle_file(url, filename=f"{catnr}.tle", reload=True)[0]
        sp = sat.at(t).subpoint()
        data[name] = (list(sp.latitude.degrees), list(sp.longitude.degrees))
    except Exception as e:
        print(f"{name} の取得に失敗: {e}")

names = list(data)
n = len(t)

# トレース: 各衛星につき [軌跡, 現在位置の点] の2本
traces = []
for i, name in enumerate(names):
    lat, lon = data[name]
    visible = (i == 0)
    traces.append(go.Scattergeo(lat=lat, lon=lon, mode="lines",
                                line=dict(color="red", width=1),
                                name="軌跡", visible=visible, showlegend=False))
    traces.append(go.Scattergeo(lat=[lat[0]], lon=[lon[0]], mode="markers",
                                marker=dict(size=10, color="yellow",
                                            line=dict(width=1, color="black")),
                                name=name, visible=visible))

# アニメーションのフレーム（全衛星の点を同時に動かす）
frames = []
for k in range(n):
    frame_data = [go.Scattergeo(lat=[data[nm][0][k]], lon=[data[nm][1][k]])
                  for nm in names]
    frames.append(go.Frame(data=frame_data,
                           traces=[2 * i + 1 for i in range(len(names))],
                           name=str(k)))

# 衛星切り替えボタン
dropdown = []
for i, name in enumerate(names):
    vis = [False] * (2 * len(names))
    vis[2 * i] = vis[2 * i + 1] = True
    dropdown.append(dict(label=name, method="update",
                         args=[{"visible": vis}]))

# --- 速度の設定 ---
# 倍率 = (step×60) ÷ (duration÷1000)  →  duration = step×60×1000 ÷ 倍率
SPEED_LIST = [10, 100, 200, 500, 1000]  # 倍率

def play_args(speed):
    duration = step * 60 * 1000 / speed
    return [None, dict(frame=dict(duration=duration, redraw=True),
                       fromcurrent=True, mode="immediate")]

speed_buttons = [
    dict(label=f"▶ {s}倍", method="animate", args=play_args(s))
    for s in SPEED_LIST
]

fig = go.Figure(data=traces, frames=frames)

fig.update_geos(projection_type="natural earth",
                showland=True, landcolor="lightgreen",
                showocean=True, oceancolor="lightblue",
                showcountries=True)

fig.update_layout(
    title=None,
    height=750,
    margin=dict(t=20, b=170),
    updatemenus=[
        # 衛星選択
        dict(type="dropdown", buttons=dropdown, direction="up",
             x=0.0, y=-0.30, xanchor="left", yanchor="top", showactive=True),
        # 速度ボタン（押すとその速度で再生開始）
        dict(type="buttons", direction="right", showactive=False,
             x=0.22, y=-0.30, xanchor="left", yanchor="top",
             buttons=speed_buttons),
        # 停止
        dict(type="buttons", showactive=False,
             x=0.85, y=-0.30, xanchor="left", yanchor="top",
             buttons=[dict(label="⏸ 停止", method="animate",
                           args=[[None], dict(frame=dict(duration=0),
                                              mode="immediate")])]),
    ],
    sliders=[dict(
        x=0.0, y=-0.05, xanchor="left", yanchor="top", len=1.0,
        steps=[dict(method="animate", label=f"+{k * step}分",
                    args=[[str(k)], dict(mode="immediate",
                                         frame=dict(duration=0, redraw=True))])
               for k in range(0, n, 30)],
        currentvalue=dict(prefix="経過: "),
    )],
)
fig.show()

[#################################] 100% 39766.tle
[#################################] 100% 60182.tle
[#################################] 100% 43672.tle
[#################################] 100% 43065.tle
[#################################] 100% 57800.tle
[#################################] 100% 41836.tle
[#################################] 100% 37158.tle
[#################################] 100% 42738.tle
[#################################] 100% 42917.tle
[#################################] 100% 25544.tle


In [3]:
!pip install skyfield plotly

from skyfield.api import load
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

SATS = {
    "だいち2号 (ALOS-2)": 39766,
    "だいち4号 (ALOS-4)": 60182,
    "いぶき2号 (GOSAT-2)": 43672,
    "しきさい (GCOM-C)": 43065,
    "XRISM": 57800,
    "ひまわり9号": 41836,
    "みちびき1号 (QZS-1)": 37158,
    "みちびき2号 (QZS-2)": 42738,
    "みちびき3号 (QZS-3)": 42917,
    "ISS": 25544,
}

ts = load.timescale()
minutes = 600
step = 1  # 1分刻み

now = ts.now().utc_datetime()
t = ts.utc(now.year, now.month, now.day, now.hour,
           range(now.minute, now.minute + minutes, step))

# 全衛星の軌跡を先に計算
data = {}
for name, catnr in SATS.items():
    try:
        url = f"https://celestrak.org/NORAD/elements/gp.php?CATNR={catnr}&FORMAT=TLE"
        #sat = load.tle_file(url)[0]
        sat = load.tle_file(url, filename=f"{catnr}.tle", reload=True)[0]
        sp = sat.at(t).subpoint()
        data[name] = (list(sp.latitude.degrees), list(sp.longitude.degrees))
    except Exception as e:
        print(f"{name} の取得に失敗: {e}")

names = list(data)
n = len(t)

# トレース: 各衛星につき [軌跡, 現在位置の点] の2本
traces = []
for i, name in enumerate(names):
    lat, lon = data[name]
    visible = (i == 0)
    traces.append(go.Scattergeo(lat=lat, lon=lon, mode="lines",
                                line=dict(color="red", width=2),
                                name="軌跡", visible=visible, showlegend=False))
    traces.append(go.Scattergeo(lat=[lat[0]], lon=[lon[0]], mode="markers",
                                marker=dict(size=14, color="yellow"),
                                name=name, visible=visible))

# アニメーションのフレーム（全衛星の点を同時に動かす）
frames = []
for k in range(n):
    frame_data = [go.Scattergeo(lat=[data[nm][0][k]], lon=[data[nm][1][k]])
                  for nm in names]
    frames.append(go.Frame(data=frame_data,
                           traces=[2 * i + 1 for i in range(len(names))],
                           name=str(k)))

# 衛星切り替えボタン
dropdown = []
for i, name in enumerate(names):
    vis = [False] * (2 * len(names))
    vis[2 * i] = vis[2 * i + 1] = True
    dropdown.append(dict(label=name, method="update",
                         args=[{"visible": vis}]))

# --- 速度の設定 ---
# 倍率 = (step×60) ÷ (duration÷1000)  →  duration = step×60×1000 ÷ 倍率
SPEED_LIST = [10, 100, 200, 500, 1000]  # 倍率

def play_args(speed):
    duration = step * 60 * 1000 / speed
    return [None, dict(frame=dict(duration=duration, redraw=True),
                       fromcurrent=True, mode="immediate")]

speed_buttons = [
    dict(label=f"▶ {s}倍", method="animate", args=play_args(s))
    for s in SPEED_LIST
]

fig = go.Figure(data=traces, frames=frames)

fig.update_geos(
    projection_type="orthographic",  # リアルな地球儀（3D風）表示
    showland=True, landcolor="darkolivegreen",
    showocean=True, oceancolor="midnightblue",
    showcountries=True, countrycolor="gray",
    showlakes=True, lakecolor="midnightblue",
)

fig.update_layout(
    title=None,
    height=750,
    margin=dict(l=0, r=0, t=20, b=170),
    updatemenus=[
        # 衛星選択
        dict(type="dropdown", buttons=dropdown, direction="up",
             x=0.0, y=-0.30, xanchor="left", yanchor="top", showactive=True),
        # 速度ボタン（押すとその速度で再生開始）
        dict(type="buttons", direction="right", showactive=False,
             x=0.22, y=-0.30, xanchor="left", yanchor="top",
             buttons=speed_buttons),
        # 停止
        dict(type="buttons", showactive=False,
             x=0.85, y=-0.30, xanchor="left", yanchor="top",
             buttons=[dict(label="⏸ 停止", method="animate",
                           args=[[None], dict(frame=dict(duration=0),
                                              mode="immediate")])]),
    ],
    sliders=[dict(
        x=0.0, y=-0.05, xanchor="left", yanchor="top", len=1.0,
        steps=[dict(method="animate", label=f"+{k * step}分",
                    args=[[str(k)], dict(mode="immediate",
                                         frame=dict(duration=0, redraw=True))])
               for k in range(0, n, 30)],
        currentvalue=dict(prefix="経過: "),
    )],
)
fig.show()

[#################################] 100% 39766.tle
[#################################] 100% 60182.tle
[#################################] 100% 43672.tle
[#################################] 100% 43065.tle
[#################################] 100% 57800.tle
[#################################] 100% 41836.tle
[#################################] 100% 37158.tle
[#################################] 100% 42738.tle
[#################################] 100% 42917.tle
[#################################] 100% 25544.tle


In [4]:
!pip install skyfield

from skyfield.api import load

CANDIDATES = {
    "ISS": 25544,
    "だいち2号": 39766,
    "だいち4号": 60182,
    "いぶき2号": 43672,
    "しきさい": 43065,
    "ひまわり9号": 41836,
    "みちびき3号": 42917,
    "XRISM": 57800,
    "IGS Optical 5": 40538,
    "IGS Optical 6": 43223,
    "IGS Optical 7": 45165,
    "IGS Optical 8": 58762,
    "IGS Radar 5": 42072,
    "IGS Radar 7": 55329,
}

for name, catnr in CANDIDATES.items():
    url = f"https://celestrak.org/NORAD/elements/gp.php?CATNR={catnr}&FORMAT=TLE"
    try:
        sats = load.tle_file(url, filename=f"{catnr}.tle", reload=True)
        print(f"○ {name} ({catnr}): {sats[0].name}")
    except Exception:
        print(f"× {name} ({catnr}): 取得できない")

[#################################] 100% 25544.tle


○ ISS (25544): ISS (ZARYA)


[#################################] 100% 39766.tle


○ だいち2号 (39766): ALOS-2


[#################################] 100% 60182.tle


○ だいち4号 (60182): ALOS-4 (DAICHI-4)


... (6 outputs left out)

[#################################] 100% 42917.tle


○ みちびき3号 (42917): QZS-3 (MICHIBIKI-3)


[#################################] 100% 57800.tle


○ XRISM (57800): XRISM
× IGS Optical 5 (40538): 取得できない
× IGS Optical 6 (43223): 取得できない
× IGS Optical 7 (45165): 取得できない
× IGS Optical 8 (58762): 取得できない
× IGS Radar 5 (42072): 取得できない
× IGS Radar 7 (55329): 取得できない
